# Stage 3 Instruction Training

Train the profile-to-protein instruction model from the configured instruction JSONL stream. The implementation lives in `libs/instruction`; reference folders stay read-only.

In [ ]:
import json
import os
import platform
import sys
from dataclasses import replace
from pathlib import Path

import torch


def find_repo_dir(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        repo_dir = Path(repo_dir_env).expanduser().resolve()
        if (repo_dir / "pyproject.toml").exists() and (repo_dir / "libs").is_dir():
            return repo_dir
    raise RuntimeError("Run this notebook from inside the repo or set MDNAC_REPO_DIR.")


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


REPO_DIR = find_repo_dir(Path.cwd())
os.chdir(REPO_DIR)
load_env_file(REPO_DIR / ".env")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.core.pretrain.training_config import _load_yaml_mapping
from libs.instruction import (
    InstructionTrainer,
    audit_instruction_jsonl,
    build_instruction_training_config,
)

print(f"REPO_DIR = {REPO_DIR}")

In [ ]:
CONFIG_PATH = REPO_DIR / "instruction.16gb.yaml"
raw_config = _load_yaml_mapping(CONFIG_PATH)
CONFIG = build_instruction_training_config(REPO_DIR, raw_config)

IS_WINDOWS = platform.system() == "Windows"
RUNNING_IN_NOTEBOOK = "ipykernel" in sys.modules
if IS_WINDOWS and RUNNING_IN_NOTEBOOK and CONFIG.num_workers != 0:
    CONFIG = replace(CONFIG, num_workers=0)
if IS_WINDOWS and RUNNING_IN_NOTEBOOK and CONFIG.multi_gpu_mode == "ddp":
    CONFIG = replace(CONFIG, multi_gpu_mode="data_parallel")

config_preview = {
    "config_path": str(CONFIG_PATH),
    "instruction_jsonl_paths": [str(path) for path in CONFIG.instruction_jsonl],
    "artifact_source_jsonl": str(CONFIG.artifact_source_jsonl),
    "base_checkpoint_path": str(CONFIG.base_checkpoint_path),
    "output_dir": str(CONFIG.output_dir),
    "artifact_dir": str(CONFIG.artifact_dir),
    "optimizer": {
        "type": CONFIG.optimizer_type,
        "learning_rate": CONFIG.learning_rate,
        "muon_learning_rate": CONFIG.muon_learning_rate,
        "weight_decay": CONFIG.weight_decay,
        "fused": CONFIG.fused,
    },
    "runtime": {
        "platform": platform.system(),
        "running_in_notebook": RUNNING_IN_NOTEBOOK,
        "cuda_device_count": torch.cuda.device_count(),
        "device": CONFIG.device,
        "multi_gpu_mode": CONFIG.multi_gpu_mode,
        "mixed_precision": CONFIG.mixed_precision,
    },
    "data": {
        "batch_size": CONFIG.batch_size,
        "gradient_accumulation_steps": CONFIG.gradient_accumulation_steps,
        "num_workers": CONFIG.num_workers,
        "shuffle_buffer_size": CONFIG.shuffle_buffer_size,
    },
}
print(json.dumps(config_preview, indent=2, ensure_ascii=False))

In [ ]:
audit = audit_instruction_jsonl(
    CONFIG.instruction_jsonl,
    default_sequence_type=CONFIG.default_sequence_type,
    instruction_field=CONFIG.instruction_field,
    input_field=CONFIG.input_field,
    output_field=CONFIG.output_field,
)
print(json.dumps(audit.to_dict(), indent=2, ensure_ascii=False))

In [ ]:
trainer = InstructionTrainer(CONFIG)
result = trainer.train()
print(json.dumps(result.to_dict(), indent=2, ensure_ascii=False))